In [1]:
import numpy as np
import pandas as pd

from constants import *
from self_consistency_k import bdg_sc_full_k

In [2]:
initial_seeds = np.array([
    [0,0,0,0],  # normal state
    # [0.1, -0.1, 0, 0],  # px 
    # [0, 0, 0.1, -0.1],  # py
    # [0.1, -0.1, 0.1, -0.1],  # px + py
    [0.1, -0.1, 0.1j, -0.1j],  # px + i*py
    # [0.1, 0.1, -0.1, -0.1],  # d-wave
    # [0.1+0.1, 0.1-0.1, -0.1, -0.1],  # d-wave + px
    # [0.1, 0.1, -0.1+0.1, -0.1-0.1],  # d-wave + py
    [0.1, 0.1, 0.1, 0.1],  # s-wave
    [0.1+0.1, 0.1-0.1, 0.1, 0.1],  # s-wave + px
    [0.1+0.1, 0.1-0.1, 0.1+0.1, 0.1-0.1],  # s-wave + px
    # [0.1, 0.1, 0.1+0.1, 0.1-0.1] # s-wave + py
], dtype=np.complex128)
seed_strings = [
    "normal state",
    # "px", "py", "px+py", 
    "px+i*py",
    # "d-wave", "d-wave+px", "d-wave+py",
    "s-wave", 
    "s-wave+px",
    "s-wave+px + py",
    # "s-wave+py"
]

In [ ]:
mu_arr = np.linspace(2.11, 2.30, 30)
print(mu_arr)
T_arr = np.linspace(0.001, 0.1, 34)
T_arr = np.linspace(0.001, 0.3, 50)

print(T_arr)

[2.11       2.11689655 2.1237931  2.13068966 2.13758621 2.14448276
 2.15137931 2.15827586 2.16517241 2.17206897 2.17896552 2.18586207
 2.19275862 2.19965517 2.20655172 2.21344828 2.22034483 2.22724138
 2.23413793 2.24103448 2.24793103 2.25482759 2.26172414 2.26862069
 2.27551724 2.28241379 2.28931034 2.2962069  2.30310345 2.31      ]
[0.001      0.00710204 0.01320408 0.01930612 0.02540816 0.0315102
 0.03761224 0.04371429 0.04981633 0.05591837 0.06202041 0.06812245
 0.07422449 0.08032653 0.08642857 0.09253061 0.09863265 0.10473469
 0.11083673 0.11693878 0.12304082 0.12914286 0.1352449  0.14134694
 0.14744898 0.15355102 0.15965306 0.1657551  0.17185714 0.17795918
 0.18406122 0.19016327 0.19626531 0.20236735 0.20846939 0.21457143
 0.22067347 0.22677551 0.23287755 0.23897959 0.24508163 0.25118367
 0.25728571 0.26338776 0.2694898  0.27559184 0.28169388 0.28779592
 0.29389796 0.3       ]


In [3]:
mu_arr = np.linspace(1.8, 4.0, 1)
T_arr = np.linspace(0.001, 0.2, 1)

mu_arr = [2.207]  #works for s + px and s + py
# mu_arr = [1]
T_arr = [0.0001]
free_tol = 0.01
print(mu_arr)
print(T_arr)

[2.207]
[0.0001]


In [4]:
t=1
V_prime=1.5
V = 1.5

h = np.array([0, 0, 0])
Nx, Ny = 75, 75

atol = 1e-5
rtol = 1e-3
maxiter=1000

In [5]:
def stable_config(out, atol=1e-6, rtol=0.1):
    stable = {}
    F_max = out.corr[np.argmax(np.abs(out.corr))]
    if np.abs(F_max) < atol:
        return stable
    for i, c in enumerate(out[1:-3]):
        if np.abs(c) > atol and np.abs(c) > rtol * np.abs(F_max):
            stable[corr_strings[i]] = c
    return stable
def same_stable_dict(d1, d2, atol=1e-6, rtol=1e-3):
    if d1.keys() != d2.keys():
        return False

    for k in d1:
        if not np.isclose(d1[k], d2[k], atol=atol, rtol=rtol):
            return False

    return True

In [6]:
print_output = True
records = []

mu = mu_arr[0]

for T in T_arr:
    best_free = np.inf
    configs = []
    print("====================================")
    print(f"mu={mu:.4f}, T={T:.4f}")
    print("====================================")
    for seed1, seed_str1 in zip(initial_seeds, seed_strings):
            print(f"  Seed: {seed_str1}")

            out = bdg_sc_full_k(
                t, mu, Nx, Ny, V=V,
                temperature=T, maxiter=maxiter,
                atol=atol,rtol=rtol,
                F_init=seed1,
            )
            if print_output == True:
                print(f"F_swave: {out.F_swave}, F_dwave: {out.F_dwave}")
                print(f"F_px: {out.F_px}, F_py: {out.F_py}")
                print(f"Free_energy = {out.free_energy}")
            stable = stable_config(out, atol=atol)

            print("------------------------------------------")
            if (out.free_energy < best_free 
                and np.abs(out.free_energy - best_free) > free_tol):

                best_free = out.free_energy
                configs = [{
                    "stable": stable,
                    "free": out.free_energy,
                }]

            elif np.abs(out.free_energy - best_free) <= free_tol:
                if len(stable) != 0 and not any(
                    same_stable_dict(c["stable"], stable, atol=atol, rtol=rtol)
                    for c in configs
                ):
                    configs.append({
                        "stable": stable,
                        "free": out.free_energy,
                    })

            print(configs)
            print("------------------------------------------")

    # Append one row per (mu, T)
    records.append({
        "mu": mu,
        "T": T,
        "best_free": best_free,
        "configs": configs
    })

# Create DataFrame
df = pd.DataFrame(records)

mu=2.2070, T=0.0001
  Seed: normal state
F_swave: 0j, F_dwave: 0j
F_px: 0j, F_py: 0j
Free_energy = -13979.941040427995
------------------------------------------
[{'stable': {}, 'free': np.float64(-13979.941040427995)}]
------------------------------------------
  Seed: px+i*py
F_swave: (1.720777210849163e-18-1.431146867680866e-17j), F_dwave: (1.748669741104451e-18-3.0357660829594124e-18j)
F_px: (0.00896574302807466+1.1058213649751904e-18j), F_py: (1.793016548104644e-18+0.008965743028074728j)
Free_energy = -13982.862254031379
------------------------------------------
[{'stable': {'F_px': np.complex128(0.00896574302807466+1.1058213649751904e-18j), 'F_py': np.complex128(1.793016548104644e-18+0.008965743028074728j)}, 'free': np.float64(-13982.862254031379)}]
------------------------------------------
  Seed: s-wave
F_swave: (0.008942806780887484+0j), F_dwave: (1.5612511283791264e-17+0j)
F_px: -1.1279567181671094e-18j, F_py: -2.967606400485043e-18j
Free_energy = -13982.853070061445
------

In [7]:
mu_arr = np.linspace(0.001, 0.6, 10)
print(mu_arr)

[0.001      0.06755556 0.13411111 0.20066667 0.26722222 0.33377778
 0.40033333 0.46688889 0.53344444 0.6       ]
